# Test the UC4P2 compare API (with retrieval)

This notebook calls the same FastAPI the cluster will expose: **disambiguate → expand → decompose → retrieve (Cross-Dataset Discovery) → synthesize**.

It uses FastAPI `TestClient`, so you do **not** need a running uvicorn first. Fill `.env` with `DG_USERNAME` / `DG_PASSWORD` so search can get a Keycloak token.

Optional: `QUERY_DISAMBIGUATION_URL` (defaults to the Scayle dev ingress).

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env", override=False)
load_dotenv(ROOT.parent / "dg-query-disambiguation" / ".env", override=False)
os.environ.setdefault("AUTH_DISABLED", "true")
os.environ.setdefault("QDMR_BACKEND", "dspy")
os.environ.setdefault("QDMR_MODEL_NAME_OR_PATH", "")

from src.llm import configure_lm_from_env, llm_endpoint, selected_model

configured = configure_lm_from_env()
print("ROOT", ROOT)
print("LLM configured:", configured)
if configured:
    print("  model:", selected_model())
    print("  endpoint:", llm_endpoint())
print("QUERY_DISAMBIGUATION_URL:", os.getenv("QUERY_DISAMBIGUATION_URL", "(default ingress)"))
print("CROSS_DATASET_DISCOVERY_URL:", os.getenv("CROSS_DATASET_DISCOVERY_URL", "(default ingress)"))
print("DG_USERNAME:", os.getenv("DG_USERNAME") or "(missing)")

## POST `/compare` in-process

`TestClient` mounts the app in this kernel. The JSON is the same payload you would get from curl or the cluster ingress.

In [ ]:
from fastapi.testclient import TestClient

from src.app.main import app

QUESTION = "How did a marriage look like in the 1800s compared to now?"

client = TestClient(app)
health = client.get("/health")
print("health", health.status_code, health.json())

response = client.post(
    "/compare",
    json={"query": QUESTION, "query_id": "nb-marriage", "k": 5},
)
print("compare", response.status_code)
body = response.json()
if response.status_code != 200:
    raise RuntimeError(body)

print("status:", body.get("status"))
print("steps:", body.get("steps"))
print("concept:", body.get("concept"), "|", body.get("comparison_type"))
if body.get("clarification"):
    print("clarification:", body["clarification"])

## Retrieval: search queries and passages

In [ ]:
print("search queries")
for slice_id, query in (body.get("search_queries") or {}).items():
    print(f"  [{slice_id}] {query}")

errors = body.get("retrieval_errors") or {}
if errors:
    print("\nretrieval errors")
    for slice_id, message in errors.items():
        print(f"  [{slice_id}] {message}")

print("\npassages")
for slice_id, items in (body.get("passages") or {}).items():
    print(f"\n=== {slice_id} ({len(items)} hits) ===")
    for item in items[:3]:
        snippet = (item.get("text") or item.get("text_snippet") or "")[:400]
        print(f"- {item.get('doc_id')}  sim={item.get('similarity')}  dataset={item.get('dataset_id')}")
        print(f"  {snippet}")

## Answer

In [ ]:
answer = body.get("synthesized_answer") or "(no answer)"
display(Markdown(answer))
print("\ncitations:", len(body.get("grounded_citations") or []))
for citation in (body.get("grounded_citations") or [])[:6]:
    print(f"  [{citation.get('citation_id')}] {citation.get('slice_id')} {citation.get('source_document')}")

## Inspect one graph node at a time

Stop after retrieval if you only want search hits.

In [ ]:
from src.orchestration.graph import stream_steps

for event in stream_steps(QUESTION, query_id="nb-steps", until="retrieve_slices"):
    name = event["step"]
    output = event["output"]
    print(f"\n===== {name} =====")
    if name == "disambiguate":
        print("status:", output.get("status"), "source:", output.get("disambiguation_source"))
        print("concept:", output.get("concept"), "type:", output.get("comparison_type"))
        print("slices:", output.get("slices"))
    elif name == "decompose":
        for step in output.get("qdmr") or []:
            print(step.get("instruction"))
    elif name == "retrieve_slices":
        print("queries:", output.get("search_queries"))
        print("errors:", output.get("retrieval_errors"))
        for slice_id, items in (output.get("passages") or {}).items():
            print(f"  {slice_id}: {len(items)} passages")
    else:
        keys = [key for key in output if key != "pipeline"]
        print("keys:", keys)

## Optional: live HTTP server

If uvicorn is already running (`PYTHONPATH=. python -m uvicorn src.app.main:app --port 8080`), this cell talks to it over HTTP instead of `TestClient`.

In [ ]:
import json
import urllib.error
import urllib.request

API = os.getenv("LANGUAGE_PILOT_URL", "http://127.0.0.1:8080").rstrip("/")
payload = json.dumps({"query": QUESTION, "query_id": "nb-http", "k": 3}).encode()
request = urllib.request.Request(
    f"{API}/compare",
    data=payload,
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    method="POST",
)
try:
    with urllib.request.urlopen(request, timeout=180) as http:
        live = json.loads(http.read().decode())
    print("live", API, "status=", live.get("status"), "steps=", live.get("steps"))
    print((live.get("synthesized_answer") or "")[:500])
except urllib.error.URLError as exc:
    print(f"No server at {API}: {exc.reason}")
    print("Start one with: PYTHONPATH=. python -m uvicorn src.app.main:app --port 8080")